In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Load variables

### Transform 

In [0]:
df = spark.sql(f"""
    SELECT
         CAST(MBRSHP_NBR AS STRING)             AS MBRSHP_NBR
        ,CAST(MBRSHP_SID AS LONG)               AS MBRSHP_SID
        ,CAST(MBRSHP_TYPE_ID AS INT)            AS MBRSHP_TYPE_ID
        ,CAST(MBRSHP_FEE_INC AS DECIMAL(10,3))  AS MBRSHP_FEE_INC
        ,CAST(MBRSHP_SUB_TYPE AS STRING)        AS MBRSHP_SUB_TYPE
        ,TO_DATE(MBRSHP_ENR_DT, "yyyy-MM-dd")   AS MBRSHP_ENR_DT
        ,TO_DATE(MBRSHP_EXP_DT, "yyyy-MM-dd")   AS MBRSHP_EXP_DT
        ,TO_DATE(MBRSHP_RNWL_DT, "yyyy-MM-dd")  AS MBRSHP_RNWL_DT
        ,CAST(RWDS_MBR_IND AS STRING)           AS RWDS_MBR_IND
        ,TO_DATE(RWDS_MBR_ENR_DT, "yyyy-MM-dd") AS RWDS_MBR_ENR_DT
        ,CAST(CLUB_OF_FREQUENCY AS INT)         AS CLUB_OF_FREQUENCY
        ,CAST(MKT_CD AS STRING)                 AS MKT_CD
        ,CAST(AUTO_RNWL_IND AS STRING)          AS AUTO_RNWL_IND
        ,TO_DATE(ER_SIGNUP_DT, "yyyy-MM-dd")    AS ER_SIGNUP_DT
        ,CAST(HOME_ZIP_CD AS STRING)            AS HOME_ZIP_CD
        ,CAST(SIC_CD AS INT)                    AS SIC_CD
        ,CAST(GRP_AFFIL_ID AS STRING)           AS GRP_AFFIL_ID
        ,CAST(HH_SID AS INT)                    AS HH_SID
        ,CAST(TRIM(PRI_SUPP_FHH_IND) AS INT)    AS PRI_SUPP_FHH_IND
    FROM 
        {bronze_master_member_extended}
""")

df.createOrReplaceTempView("source")

In [0]:
validations.validate_table(
        spark, "intermediate", 'member_extended', config_validation, df, stats_etl_path
    )

### Merge

In [0]:
df.write.mode("overwrite").saveAsTable(silver_master_member_extended)

if archive_flag:
    save_archive(df, silver_master_member_extended_archive, run_as_date)